In [ ]:
from preprocess_pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import polars as pl

In [2]:
df = pl.read_csv("data/clean.csv")
Train, Test = train_test_split(df, test_size=0.2)

In [3]:
pipeline = Pipeline()

X_train, y_train = pipeline.run_train_preprocessing(Train)
X_test, y_test = pipeline.run_inference_pipeline(Test)

In [4]:
model1 = XGBRegressor(
    n_estimators=500,          # Number of trees (boosting rounds)
    learning_rate=0.05,        # Step size shrinkage (eta)
    max_depth=6,               # Maximum depth of a tree
    subsample=0.8,             # Fraction of samples to train each tree
    colsample_bytree=0.8,      # Fraction of features to train each tree
    random_state=42,           # Ensures reproducible results
    n_jobs=-1
)
model1.fit(X_train, y_train)
y_pred = model1.predict(X_test)

In [5]:
# report

def regression_report(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Regression Report")
    print("-" * 30)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"R²   : {r2:.4f}")
    print(f"Average Error: {mae / y_test.to_numpy().mean():.2f}%")

regression_report(y_test, y_pred)


Regression Report
------------------------------
MAE  : 65033.0781
MSE  : 13856643072.0000
R²   : 0.8947
Average Error: 0.12%


In [10]:
pl.DataFrame({"Features": X_train.columns,
              "Importance": model1.feature_importances_}).sort("Importance", descending=True)

Features,Importance
str,f32
"""grade""",0.357156
"""waterfront""",0.176663
"""median_price""",0.144362
"""PC1""",0.108497
"""view""",0.05417
…,…
"""median_condition""",0.008651
"""PC3""",0.008432
"""median_grade""",0.008289


In [6]:
train_preds = model1.predict(X_train)
regression_report(y_train, train_preds)

Regression Report
------------------------------
MAE  : 40941.4258
MSE  : 3273019904.0000
R²   : 0.9757
Average Error: 0.08%


In [7]:
X_mini_test = pl.read_csv("https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv")

test_processed = pipeline.run_inference_pipeline(X_mini_test)[0]
holdout_preds = model1.predict(test_processed)
pl.DataFrame(data=holdout_preds, schema=["price"]).write_csv("data/preds.csv")